# Geomorphica Ad Generator — Colab launch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Geomorphica/Ads_generator/blob/main/geomorphica_ad-gen_launch.ipynb)

**Workflow:** keep this notebook on Google Drive, or open it from GitHub with the badge above.  
When you run it, Colab **clones the app from GitHub** and installs packages. You do not need the full project folder on Drive.

- Repo: https://github.com/Geomorphica/Ads_generator
- No tunnel signup / no IP password
- Keep the last cell running while you use the app
- Outputs are temporary on Colab — use **Download** in the app before you stop the runtime


## 1) Install core packages

Installs Streamlit and Pillow (safe to re-run).

In [ ]:
import importlib
import subprocess
import sys

def _pip_install(*pkgs: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

needed = {
    "streamlit": "streamlit>=1.28.0",
    "PIL": "Pillow>=10.0.0",
}
missing = []
for mod, spec in needed.items():
    try:
        importlib.import_module(mod)
        print(f"OK: {mod}")
    except ImportError:
        missing.append(spec)

if missing:
    print("Installing:", ", ".join(missing))
    _pip_install(*missing)
else:
    print("Core packages already installed.")

import streamlit
from PIL import Image  # noqa: F401
print("streamlit", streamlit.__version__)
print("Ready.")

## 2) Download the app from GitHub

Clones https://github.com/Geomorphica/Ads_generator.git into this Colab session (code, fonts, logos). Safe to re-run — it will `git pull` if already cloned.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Geomorphica/Ads_generator.git"
APP_DIR = Path("/content/Ads_generator")

if (APP_DIR / ".git").is_dir():
    print("Repo already present — pulling latest …")
    subprocess.check_call(["git", "-C", str(APP_DIR), "pull", "--ff-only"])
else:
    if APP_DIR.exists():
        raise SystemExit(f"{APP_DIR} exists but is not a git clone. Delete it and re-run.")
    print("Cloning", REPO_URL)
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(APP_DIR)])

os.chdir(APP_DIR)
print("Working directory:", os.getcwd())

assert (APP_DIR / "app.py").is_file(), "Missing app.py — push the app files to GitHub first."
assert (APP_DIR / "assets" / "fonts").is_dir(), "Missing assets/fonts — include Roboto fonts in the repo."

req = APP_DIR / "requirements.txt"
if req.is_file():
    print("Installing from requirements.txt …")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])

print("App folder OK.")

## 3) Start Streamlit + Cloudflare tunnel

Wait for a clickable `https://….trycloudflare.com` link. Keep this cell running.

Generated files land in `/content/Ads_generator/output/` on this Colab machine — **download them in the app** before you stop the runtime.

In [ ]:
import os
import re
import subprocess
import time
from pathlib import Path

from IPython.display import Markdown, display

APP_DIR = Path("/content/Ads_generator")
os.chdir(APP_DIR)
PORT = 8501
CF = Path("/content/cloudflared")

if not CF.is_file():
    print("Downloading cloudflared …")
    subprocess.check_call([
        "wget", "-q", "-O", str(CF),
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    ])
    CF.chmod(0o755)

log_path = Path("/content/streamlit.log")
log_f = open(log_path, "w")
st_proc = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", str(PORT),
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
    ],
    cwd=str(APP_DIR),
    stdout=log_f,
    stderr=subprocess.STDOUT,
)
print("Streamlit PID:", st_proc.pid)
time.sleep(4)

cf_proc = subprocess.Popen(
    [str(CF), "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

url_re = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
public_url = None
print("Waiting for Cloudflare URL …")
assert cf_proc.stdout is not None
for line in cf_proc.stdout:
    line = line.rstrip()
    if line:
        print(line)
    m = url_re.search(line)
    if m and public_url is None:
        public_url = m.group(0)
        display(Markdown(f"### Open the app: [{public_url}]({public_url})"))
        print("\nKeep this cell running. Download outputs in the app before stopping the runtime.\n")

cf_proc.wait()